In [0]:
import sys
from pyspark.sql import functions as F

dbutils.widgets.removeAll()
dbutils.widgets.text("run_date", "", "Run Date (YYYY-MM-DD)")

sys.path.append("/Workspace/Shared")
import aws_utils

env = aws_utils.get_env_prefix()

business_date_param = dbutils.widgets.get("run_date").strip()

if business_date_param:
    run_date = business_date_param
else:
    run_date = str(
        spark.table(f"`{env}-cdh`.primary.product")
        .agg(F.max("BusinessDate"))
        .first()[0]
    )

In [0]:

df_product = (
    spark.table(f"`{env}-cdh`.primary.product")
    .filter(F.col("BusinessDate") == run_date)
)


df_mod_src = spark.table(f"`{env}-cdh`.landed.params_loan_product_mod")
max_bd_mod = df_mod_src.agg(F.max("BusinessDate")).first()[0]
df_mod = (
    df_mod_src
    .filter(F.col("BusinessDate") == F.lit(max_bd_mod))
    .select(
        F.col("prft_product_code"),
        F.col("Modification_type").alias("ModificationType")
    )
)

df_result = (
    df_product.alias("p")
    .join(
        df_mod.alias("m"),
        F.col("p.ProductCode").cast("int") == F.col("m.prft_product_code"),
        "left"
    )
    .select(
        F.col("p.ProductId").alias("productid"),
        F.col("p.ProductKey").alias("productkey"),
        F.col("p.ProductDescription").alias("description"),
        F.col("p.SubProductCode"),
        F.col("p.SubProduct").alias("SubProductDesc"),
        F.col("m.ModificationType"),
        F.col("p.ShadowFlag"),
        F.col("p.SyndicatedFlag"),
        F.col("p.L3869Flag"),
        F.col("p.OCWFlag"),
        F.col("p.BondedFlag"),
        F.col("p.BusinessDate").alias("datadate"),
    )
)

In [0]:


(
    df_result.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"`{env}-cdh-delivery`.`it`.`CepalBILive_ref_LoanProduct`")
)